# py-nusantara Data Science Demo

This notebook demonstrates how to use the `py-nusantara` library directly in a Python data science workflow (such as inside Jupyter, Google Colab, or VS Code) without requiring any database connections.

### 1. Setup & Imports

In [1]:
import py_nusantara as nus
import pandas as pd

### 2. Direct Data Access (No Database Required)

We can query and browse the administrative regions directly from memory. The package streams the gzipped CSV files in the background.

In [2]:
# Fetch all provinces
provinces = nus.provinces()
print(f"Total Provinces: {len(provinces)}")

# Find a specific province by ID (Aceh = '11')
aceh = nus.find_province("11")
print(f"Province Found: {aceh.name} (Capital: {aceh.capital})")
print(f"Population: {aceh.population:,} | Area: {aceh.area:,} km²")

Total Provinces: 38
Province Found: Aceh (Capital: Banda Aceh)
Population: 5,623,479 | Area: 56,835.019 km²


### 3. Traversal (Province -> Regencies -> Districts -> Villages)

Relationships are loaded dynamically and traverse downstream levels smoothly. Village loading is **lazy-loaded & partitioned** by Province ID to minimize memory overhead.

In [3]:
# Traverse regencies
regencies = aceh.regencies
print(f"Regencies in {aceh.name}: {len(regencies)}")

first_regency = regencies[0]
print(f"First Regency: {first_regency.name} (ID: {first_regency.id})")

# Traverse districts
districts = first_regency.districts
print(f"Districts in {first_regency.name}: {len(districts)}")

first_district = districts[0]
print(f"First District: {first_district.name} (ID: {first_district.id})")

# Traverse villages (partitioned load)
villages = first_district.villages
print(f"Villages in {first_district.name}: {len(villages)}")
if villages:
    print(f"First Village: {villages[0].name} (Postal Code: {villages[0].postal_code})")

Regencies in Aceh: 23
First Regency: Kabupaten Aceh Selatan (ID: 1101)
Districts in Kabupaten Aceh Selatan: 18
First District: Bakongan (ID: 110101)
Villages in Bakongan: 7
First Village: Keude Bakongan (Postal Code: 23773)


### 4. Substring Search

Search administrative names case-insensitively across all levels. Scanning is optimized and terminates once limits are met.

In [4]:
results = nus.search("Bakongan")
for level, records in results.items():
    if records:
        print(f"\nMatches in [{level.upper()}]:")
        for r in records:
            print(f"  - ID: {r.id} | Name: {r.name}")


Matches in [DISTRICTS]:
  - ID: 110101 | Name: Bakongan
  - ID: 110115 | Name: Bakongan Timur

Matches in [VILLAGES]:
  - ID: 1101012001 | Name: Keude Bakongan


### 5. On-Demand Geographic Boundaries (GIS)

Download shapefiles on-demand from the CDN and verify their integrity dynamically using SHA-256 checksums.

In [5]:
# Enable the boundary column in configuration dynamically
nus.init({
    "columns": {
        "provinces": {
            "boundary": {"name": "boundary", "enabled": True}
        }
    }
})

print("Downloading provinces boundary...")
nus.download_boundaries(levels="provinces")

[PosixPath('/home/wawanwidiantara/.cache/py-nusantara/provinces.csv.gz')]

In [6]:
# Re-fetch the province and check boundary
aceh_boundary = nus.find_province("11")
print("Raw JSON coordinates (truncated):")
print(aceh_boundary.boundary[:150] + "...")

# Format to Well-Known Text (WKT) standard for spatial databases
wkt = nus.json_to_wkt(aceh_boundary.boundary)
print("\nWKT Format (truncated):")
print(wkt[:150] + "...")

Raw JSON coordinates (truncated):
[[[2.079098,97.077157],[2.090249,97.083166],[2.09145,97.098615],[2.107232,97.09295],[2.113579,97.099474],[2.110491,97.124879],[2.086132,97.154405],[2....

WKT Format (truncated):
POLYGON((97.077157 2.079098, 97.083166 2.090249, 97.098615 2.09145, 97.09295 2.107232, 97.099474 2.113579, 97.124879 2.110491, 97.154405 2.086132, 97....


### 6. Pandas DataFrames Export

Convert results to Pandas DataFrames for quick data processing and visualization.

In [7]:
# Export all provinces as DataFrame
df_provinces = nus.provinces_df()
df_provinces.head()

,id,name,capital,latitude,longitude,elevation,timezone,area,population,boundary
0,11,Aceh,Banda Aceh,5.570547,95.340809,11.0,7,56835.019,5623479,"[[[2.079098,97.077157],[2.090249,97.083166],[2..."
1,12,Sumatera Utara,Medan,3.580630,98.672000,32.0,7,72437.755,15640905,"[[[-0.635433,98.521413],[-0.633982,98.505182],..."
2,13,Sumatera Barat,Padang,-0.937670,100.360444,10.0,7,42107.674,5820359,"[[[-3.275582,100.56959],[-3.260329,100.565641]..."
3,14,Riau,Pekanbaru,0.517710,101.445840,20.0,7,89900.780,7099297,"[[[0.791175,103.09059],[0.79701,103.097113],[0..."
4,15,Jambi,Jambi,-1.603998,103.583580,28.0,7,49023.037,3834439,"[[[-1.677485,101.122559],[-1.678093,101.125664..."


In [8]:
# Export regencies in Province 11 (Aceh) as DataFrame
df_regencies = nus.regencies_df("11")
df_regencies.head()

,id,province_id,name,capital,latitude,longitude,elevation,timezone,area,population
0,1101,11,Kabupaten Aceh Selatan,Tapak Tuan,3.254802,97.174112,19.0,7,4174.211,239629
1,1102,11,Kabupaten Aceh Tenggara,Kutacane,3.473360,97.819243,168.0,7,4179.123,235589
2,1103,11,Kabupaten Aceh Timur,Idi Rayeuk,4.932737,97.783844,10.0,7,5409.158,461391
3,1104,11,Kabupaten Aceh Tengah,Takengon,4.622057,96.845668,1262.0,7,4468.417,232606
4,1105,11,Kabupaten Aceh Barat,Meulaboh,4.153010,96.131441,5.0,7,2782.606,209220
